In [1]:
import os
import snowflake.connector
from snowflake.connector.errors import Error

def test_snowflake_connection() -> int:
    required_vars = [
        "SNOWFLAKE_ACCOUNT",
        "SNOWFLAKE_USER",
        "SNOWFLAKE_TOKEN",
        "SNOWFLAKE_WAREHOUSE",
        "SNOWFLAKE_DATABASE",
        "SNOWFLAKE_SCHEMA",
        "SNOWFLAKE_ROLE",
    ]

    missing = [v for v in required_vars if not os.getenv(v)]
    if missing:
        print(f"Variables manquantes: {', '.join(missing)}")
        return 1

    conn = None
    try:
        connect_kwargs = {
            "account": os.environ["SNOWFLAKE_ACCOUNT"],
            "user": os.environ["SNOWFLAKE_USER"],
            "password": os.environ["SNOWFLAKE_TOKEN"],
            "warehouse": os.environ["SNOWFLAKE_WAREHOUSE"],
            "database": os.environ["SNOWFLAKE_DATABASE"],
            "schema": os.environ["SNOWFLAKE_SCHEMA"],
            "role": os.environ["SNOWFLAKE_ROLE"],
            "login_timeout": 10,
            "network_timeout": 10,
        }

        authenticator = os.getenv("SNOWFLAKE_AUTHENTICATOR")
        if authenticator:
            connect_kwargs["authenticator"] = authenticator

        conn = snowflake.connector.connect(**connect_kwargs)

        with conn.cursor() as cur:
            cur.execute("SELECT CURRENT_VERSION(), CURRENT_USER(), CURRENT_ROLE()")
            version, user, role = cur.fetchone()
            print("✅ Connexion Snowflake OK")
            print(f"Version    : {version}")
            print(f"Utilisateur: {user}")
            print(f"Rôle       : {role}")
        return 0

    except Error as e:
        message = str(e)
        if "Network policy is required" in message:
            print("❌ Connexion refusée: une Network Policy est obligatoire sur ce compte.")
            print("Actions à faire:")
            print("  1) Demander à l'admin Snowflake d'autoriser votre IP/réseau dans la policy.")
            print("  2) Vérifier si un authenticator SSO est requis (export SNOWFLAKE_AUTHENTICATOR=externalbrowser).")
            print("  3) Vérifier que SNOWFLAKE_ACCOUNT est correct (sans https:// ni .snowflakecomputing.com).")
            return 4

        print(f"❌ Échec connexion Snowflake: {message}")
        return 2

    except Exception as e:
        print(f"❌ Erreur inattendue: {e}")
        return 3

    finally:
        if conn is not None:
            conn.close()

result = test_snowflake_connection()
print(f"Code retour: {result}")

✅ Connexion Snowflake OK
Version    : 10.7.1
Utilisateur: CAMELIA.MAZOUZ
Rôle       : ACCOUNTADMIN
Code retour: 0


In [ ]:
import os
print(os.getenv("MISTRAL_API_KEY"))

In [ ]:
from langchain_openai import ChatOpenAI
import dotenv
dotenv.load_dotenv()


llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=1,
)
prompt = "Quelle est la capitale de la France ?"
# Utilise la variable `prompt` déjà définie dans le notebook
reponse = llm.invoke(prompt)
print(reponse.content)

# Exemple avec une autre question
# reponse2 = llm.invoke("Explique la photosynthèse en 2 phrases.")
# print(reponse2.content)


In [4]:
!pip3 install mistralai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [mistralai]12 [mistralai]try-api]

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [5]:
from mistralai import Mistral

api_key = "uYjtS4EklNWuYR87xB0KLXNiIqxcT4mx"
client = Mistral(api_key=api_key)

response = client.chat.complete(
    model="mistral-small-latest",
    messages=[
        {"role": "user", "content": "Quelle est la capitale de la France ?"}
    ]
)

print(response.choices[0].message.content)

La capitale de la France est **Paris**.

C'est une ville emblématique, connue pour ses monuments comme la Tour Eiffel, le Louvre, Notre-Dame et bien d'autres. Paris est également un centre culturel, économique et politique majeur en Europe.

Tu veux des informations supplémentaires sur Paris ? 😊


In [10]:
import snowflake.connector
import dotenv, os
dotenv.load_dotenv()

conn = snowflake.connector.connect(
    account=os.environ["SNOWFLAKE_ACCOUNT"],
    user=os.environ["SNOWFLAKE_USER"],
    password=os.environ["SNOWFLAKE_TOKEN"],
    warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
    database=os.environ["SNOWFLAKE_DATABASE"],
    schema=os.environ["SNOWFLAKE_SCHEMA"],
    role=os.environ.get("SNOWFLAKE_ROLE", "ACCOUNTADMIN"),
)

sql_steps = [
    # 1. Activer l'inférence cross-région (nécessaire si le modèle n'est pas dispo localement)
    "ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION'",

    # 2. Créer le Cortex Search Service
    """
    CREATE OR REPLACE CORTEX SEARCH SERVICE SUPPORT_TICKETS_SEARCH_SERVICE
    ON body_answer
    WAREHOUSE = {warehouse}
    TARGET_LAG = '1 hour'
    AS (
        SELECT
            ANSWER   AS body_answer,
            SUBJECT  AS question,
            TYPE,
            PRIORITY,
            LANGUAGE
        FROM PROJECT_DB.PUBLIC.SUPPORT_TICKETS
    )
    """.format(warehouse=os.environ["SNOWFLAKE_WAREHOUSE"]),
]

with conn.cursor() as cur:
    for sql in sql_steps:
        print(f"Execution: {sql.strip()[:80]}...")
        cur.execute(sql)
        print(f"  → {cur.fetchone()}\n")

conn.close()
print("✅ Cortex Search Service créé avec succès.")


Execution: ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION'...
  → ('Statement executed successfully.',)

Execution: CREATE OR REPLACE CORTEX SEARCH SERVICE SUPPORT_TICKETS_SEARCH_SERVICE
    ON bo...
  → ('Cortex search service SUPPORT_TICKETS_SEARCH_SERVICE successfully created.',)

✅ Cortex Search Service créé avec succès.
